In [ ]:
# ========== 导入：多模态视觉描述（LLaVA）所需工具 ==========

# 导入 ollama：调用本地 Ollama 上的多模态模型（如 llava:7b-v1.6）
import ollama
# 导入 base64：把图片二进制编码成 Base64 字符串，方便塞进模型请求
import base64
# 导入标准库 os：规范化路径、判断文件是否存在
import os


In [ ]:
# ========== encode_image：本地图片 → Base64 文本 ==========

# 定义函数：读取图片文件并返回 UTF-8 的 Base64 字符串
def encode_image(image_path):
    # 以二进制模式打开图片文件
    with open(image_path, 'rb') as f:
        # 读全部字节 → Base64 编码 → 解码成普通字符串返回
        return base64.b64encode(f.read()).decode('utf-8')


In [ ]:
# ========== 可选调试：手动指定一张图片并预览 Base64 前缀（默认注释掉） ==========

# 示例 Windows 路径（保留原文，勿改）；取消注释可本地试跑
# image_path = r"C:\Users\LAKSHYA\OneDrive\Pictures\Camera Roll\WIN_20250614_02_46_47_Pro.jpg"
# 调用 encode_image 得到 Base64
# image_base64 = encode_image(image_path)
# 只打印前 100 个字符，确认编码成功、避免刷屏
# print(image_base64[:100]) 


In [ ]:
# ========== 全局图片列表：存放若干张图的 Base64 ==========

# 空列表起步；后续 put_image() 会往里 append
image_list = []


In [ ]:
# ========== put_image：交互式读入一张本地图片并加入 image_list ==========

# 定义函数：向用户要路径，校验存在后编码并追加
def put_image():
    # 声明使用模块级 image_list（跨单元格共享）
    global image_list
    # input 提示文案保持英文原样（影响运行时交互）
    user_input_image = input("Enter image path or press enter to skip: ").strip()
    
    # 空输入：跳过加图，直接返回当前列表
    if not user_input_image:
        # 打印提示（文案保持原样）
        print("No image inserted")
        return image_list

    # normpath：统一路径分隔符（Windows/Unix）
    image_path = os.path.normpath(user_input_image)
    
    # 路径不存在：提示后递归再问一次
    if not os.path.exists(image_path):
        # 错误提示文案保持原样
        print("Image path not found! Try again or enter to leave blank")
        # 递归重试（逻辑保持原样）
        return put_image()  # Continue to allow more inputs
        



        
    # 文件存在：编码为 Base64
    image_base64 = encode_image(image_path)
    # 追加到全局列表，供后续 ollama.generate(..., images=...) 使用
    image_list.append(image_base64)
    
    # 以下为预留：按扩展名推断 MIME（当前未启用，逻辑保持注释状态）
    # Detect file extension for MIME type
    # ext = os.path.splitext(image_path)[-1].lower()
    # mime_type = 'image/jpeg' if ext in ['.jpg', '.jpeg'] else 'image/png'  # Extend if needed


    # 返回更新后的列表
    return image_list
    
    # return f"data:{mime_type};base64,{image_base64[:100]}"


In [ ]:
# ========== 初始 system 风格 prompt：面向视障用户的图像描述助手 ==========

# 注意：发给模型的英文指令字符串必须原样保留（改译会改变模型行为）
prompt=  ("System prompt: (You are a compassionate and intelligent visual assistant designed to help people who are blind or visually impaired. "
    "Your job is to look at an image and describe it in a way that helps the user understand the scene clearly. "
    "Use simple, descriptive language and avoid technical terms. Describe what is happening in the image, people's body language, clothing, facial expressions, objects, and surroundings. "
    "Be vivid and precise, as if you are painting a picture with words. "
    "Also, take into account any personal instructions or questions provided by the user—such as describing a specific person, activity, or object. "
    "If the user includes a specific prompt, prioritize that in your description.)")


In [ ]:
# ========== put_prompt：交互式追加用户问题到全局 prompt ==========

# 定义函数：读入用户输入，拼到对话历史字符串上
def put_prompt():
    # 使用并修改模块级 prompt
    global prompt
    # input 提示保持英文原样
    user_input = input("Put new prompt: ")
    # 空输入则提示并递归重问
    if not user_input:
        # 提示文案保持原样
        print("please enter a prompt")
        return put_prompt()
    # 把用户句追加到历史（前缀 "User: "）
    prompt += "\nUser: " + user_input
    # 返回更新后的完整 prompt
    return prompt


In [ ]:
# ========== image_description：加图 + 提问 + 流式调用 LLaVA ==========

# 定义函数：一次完整的「看图描述」流程
def image_description():
    # 需要读写全局 prompt（对话记忆）
    global prompt

    # 先让用户选图（可能跳过）
    put_image()
    # 没有任何图则直接返回提示字符串（文案保持原样）
    if not image_list: 
        return "No images available. Skipping..."

    # 再收集用户自然语言问题，得到完整 prompt
    user_prompt = put_prompt()
    # 累积流式输出的完整回答
    full_answer = ""

    # ollama.generate：多模态生成；stream=True 边生成边返回 chunk
    for chunk in ollama.generate(
        # 模型 id 保持原样：llava:7b-v1.6（需本机 ollama pull）
        model='llava:7b-v1.6',
        # 文本侧 prompt（含 system 说明 + 用户问题）
        prompt=user_prompt,
        # 图像侧：Base64 列表
        images=image_list,
        # 流式：逐块拿到 response 字段
        stream=True
    ):
        # 从 chunk 字典取增量文本，缺省为空串
        content = chunk.get("response", "")
        # 实时打印到控制台（end="" 不换行；flush 立即刷出）
        print("\n\n Final Answer:",content, end="", flush=True)  # Live stream to console
        # 拼进完整答案
        full_answer += content

    # 把本轮 User/Assistant 写回全局 prompt，形成简易多轮记忆
    prompt += "\nUser: " + user_prompt + "\nAssistant: " + full_answer
    # 返回完整回答字符串
    return full_answer


In [ ]:
# ========== call_llava：清空图片列表后连续跑若干轮描述 ==========

# 定义入口函数：演示多轮迭代
def call_llava():
    # 清空全局 image_list，避免上一轮残留
    image_list.clear()
    # 固定循环 5 次（逻辑保持原样）
    for i in range(5):
        # 打印当前轮次（文案保持原样）
        print(f"\n Iteration {i+1}")
        # 跑一轮：加图 → 提问 → 流式生成
        answer = image_description()
        # 再打印一次完整最终答案
        print("\n\n Final Answer:", answer)
    


In [ ]:
# ========== 启动第一部分演示：循环调用 LLaVA 看图描述 ==========

# 直接调用；笔记本里会阻塞在 input() 等待路径/问题
call_llava()


# 第 2 周进阶：用「工具调用」让助手更聪明

## 练习目标

在第一部分「直接把图片丢给 LLaVA」的基础上，进一步练习第 2 周的核心概念：**工具（Tools）**：

- 模型先判断「要不要看图」
- 若需要，输出结构化的 `TOOL_CALL: analyze_image(...)`
- 程序解析该字符串，真正调用 `analyze_image`，再把结果回填对话

## 和本课的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多模态模型 | Ollama 上的 `llava:7b-v1.6` |
| Tool / Function Calling（简易版） | 用正则解析 `TOOL_CALL:` 文本协议 |
| 对话 messages | `system` / `user` / `function` 角色拼接 |

## 怎么跑

1. 确保本机 Ollama 已拉取 `llava:7b-v1.6`
2. 从上到下运行：先定义 `process_response` / `analyze_image`，再跑 `chat_loop`
3. 按提示输入图片路径与问题；观察模型是否触发工具调用


In [ ]:
# ========== 对话 messages：给「带工具的聊天」准备空历史 ==========

# Chat Completions 风格的消息列表；后面会 append system/user/function
messages = []


In [ ]:
# ========== system_content：规定何时直接回答、何时发出 TOOL_CALL ==========

# 发给模型的系统指令（英文原文必须保留，改译会改变工具协议行为）
system_content = (
    "You are a helpful assistant for visually impaired users. "
    "You are capable of answering questions directly or calling a function to analyze an image if needed. "
    "There is a list of images available, indexed from 0. "
    "When a user asks a question, first determine whether any image in the list is needed to answer. "
    "If yes, reply in this structured format:\n\n"
    "TOOL_CALL: analyze_image(<image_index_or_range>, prompt='<description_request>')\n\n"
    "If image is not needed, just answer the user directly in plain natural language.\n"
    "Be clear and use descriptive but accessible language suitable for blind users."
)


In [ ]:
# ========== 把 system 指令写入 messages 列表 ==========

# role=system：全局行为约束；content 用上一格的 system_content
messages.append({"role":"system","content":system_content})


In [ ]:
# ========== chat_loop：单轮「加图 → 提问 → 解析工具调用」主循环 ==========

# 定义主交互函数（单轮版本；文档字符串改为中文教学说明）
def chat_loop():
    """主聊天循环（单轮版）：收集图片与问题，调用 LLaVA，再交给 process_response 处理工具调用。"""
    # 读写全局图片列表与对话历史
    global image_list, messages
    
    # 打印分隔线与标题（展示文案保持英文原样）
    print("\n" + "="*50)
    print("LLaVA Assistant for Visually Impaired Users")
    print("="*50 + "\n")
    
    # 步骤 1：可选地加载图片到 image_list
    print("Step 1: Add images (optional)")
    put_image()
    # 再追加一条 system，告诉模型当前有几张图、合法下标范围
    messages.append({
        "role": "system", 
        "content": f"There are {len(image_list)} images available (index 0-{len(image_list)-1})."
    })
    
    # 步骤 2：收集用户问题并写入 messages
    print("\nStep 2: Ask a question about the images")
    user_content = put_prompt()
    messages.append({"role": "user", "content": user_content})
    
    # 调用本地多模态聊天；失败则打印异常
    try:
        # ollama.chat：非流式拿完整 message.content
        response = ollama.chat(
            model='llava:7b-v1.6',
            messages=messages
        )["message"]["content"]
        # 先打印模型原始回复
        print("assistant: ",response)    
        # 再走工具协议解析（可能真正调用 analyze_image）
        processed_response = process_response(response)
        # 打印处理后的最终助手输出
        print(f"\nASSISTANT: {processed_response}\n")
        
    except Exception as e:
        # 异常信息前缀保持原样
        print(f"Error occurred: {e}")
    
    # 会话结束提示（文案保持原样）
    print("\nSession ended. Goodbye!")


In [ ]:
# ========== process_response：解析 TOOL_CALL 或把普通回答记入历史 ==========

# 定义函数：若是工具调用则执行并格式化结果；否则原样返回
def process_response(response):
    """处理模型回复：识别 TOOL_CALL 协议，校验图片下标，调用 analyze_image。"""
    # 以 TOOL_CALL: 开头则进入工具分支
    if response.strip().startswith("TOOL_CALL:"):
        # 正则：捕获下标/切片表达式，以及 prompt='...' 里的描述请求（依赖此前环境已 import re）
        pattern = r"TOOL_CALL:\s*analyze_image\((.*?)\s*,\s*prompt='(.*?)'\)"
        # DOTALL：让 . 也能匹配换行
        match = re.search(pattern, response, re.DOTALL)
        
        # 格式不对：记一条 assistant 错误并返回
        if not match:
            error_msg = "Error: Invalid TOOL_CALL format."
            messages.append({"role": "assistant", "content": error_msg})
            return error_msg
            
        # group(1)=下标表达式；group(2)=用户描述请求
        image_expr = match.group(1).strip()
        prompt = match.group(2).strip()
        
        try:
            # 支持 "1:3" 切片或单个整数下标
            if ":" in image_expr:  # Range (e.g., "1:3")
                # 拆成 start/end，再 list(range(...))
                start, end = map(int, image_expr.split(":"))
                index_or_range = list(range(start, end))
            else:  # Single index
                # 单个下标转 int
                index_or_range = int(image_expr)
                
            # 合法下标上界
            max_index = len(image_list) - 1
            # 列表下标：任一越界则报错
            if isinstance(index_or_range, list):
                if any(i < 0 or i > max_index for i in index_or_range):
                    error_msg = f"Error: Image index out of range (0-{max_index})."
                    messages.append({"role": "assistant", "content": error_msg})
                    return error_msg
            # 单个下标越界
            elif index_or_range < 0 or index_or_range > max_index:
                error_msg = f"Error: Image index out of range (0-{max_index})."
                messages.append({"role": "assistant", "content": error_msg})
                return error_msg
                
            # 真正调用视觉分析函数
            result = analyze_image(index_or_range, prompt)
            # 调试打印（原文拼写保持原样）
            print("funtion called")
            # 以 function 角色把工具结果写回 messages
            messages.append({
                "role": "function",
                "name": "analyze_image",
                "content": result
            })
            
            # 包装成可读的分析结果文本返回
            formatted_result = f"\nIMAGE ANALYSIS RESULT:\n{result}"
            return formatted_result

        except Exception as e:
            # 解析/执行异常：记入历史并返回
            error_msg = f"Error processing TOOL_CALL: {e}"
            messages.append({"role": "assistant", "content": error_msg})
            return error_msg
    else:
        # 非工具调用：普通助手回复，直接入历史
        messages.append({"role": "assistant", "content": response})
        return response


In [ ]:
# ========== analyze_image：按索引取图，流式调用 LLaVA 做无障碍描述 ==========

# 定义工具函数：供 process_response 在 TOOL_CALL 时调用
def analyze_image(index_or_range, prompt):
    """对指定下标（或下标列表）的图片调用 LLaVA，生成面向视障用户的描述。"""
    # 读取全局图片 Base64 列表
    global image_list
    
    # 单个 int：取一张；list：取多张；其它类型报错
    if isinstance(index_or_range, int):
        images = [image_list[index_or_range]]
    elif isinstance(index_or_range, list):
        images = [image_list[i] for i in index_or_range]
    else:
        return "Invalid image index/range specified."
    
    # 空列表防护
    if not images:
        return "No images available for analysis."
    
    # 组合给 LLaVA 的英文描述指令 + 用户具体请求（字符串保持原样）
    full_prompt = (
        "Describe the image clearly for a visually impaired user. "
        "Be detailed about objects, people, colors, spatial relationships, "
        "and any important context. "
        f"User's specific request: {prompt}"
    )
    
    # 累积流式输出
    output = ""
    try:
        # 再次流式 generate；只传选中的 images 子集
        for chunk in ollama.generate(
            model='llava:7b-v1.6',
            prompt=full_prompt,
            images=images,
            stream=True
        ):
            # 拼接每个 chunk 的 response 字段
            output += chunk.get('response', "")
    except Exception as e:
        # 分析失败时返回错误字符串（前缀保持原样）
        return f"Error analyzing image: {e}"
    
    # 成功则返回完整描述文本
    return output


In [ ]:
# ========== 启动第二部分：多轮 chat_loop（工具调用版） ==========

# 注意：此处原代码是 image_list.clear（未加括号），逻辑保持原样不改动
image_list.clear
# 连续进入 5 次单轮会话（每次内部仍会 input）
for i in range(5):
    chat_loop()
